# LLM Laptop Preference Analysis: Scalable Personalization

## 1. Introduction & Methodology

Having validated the GRUM pipeline in the discrete Colors domain, we now move to a more complex, multi-featured domain: **Laptops**.

### Domain: Laptops
The dataset consists of 12 laptops ranging from budget-friendly options ($450) to high-end professional hardware ($2800). Each laptop is defined by 8 features: Brand (Apple, Dell, HP, Lenovo, ASUS), CPU Rank, RAM size, and Price.

### Agent Setup: Prompt Contexts (Personas)
We use 4 archetypal prompt contexts to test personalization:
- **Student**: Focused on value and basic functionality.
- **Gamer**: Prioritizes high RAM, GPU, and processing power.
- **Business**: Values portability and brand reputation (Dell, Lenovo).
- **Editor**: Needs maximum rendering power (MacBook Pro, high RAM).

In this analysis, we examine results using **Hidden State (HS)** embeddings. Note: Hybrid one-hot runs were omitted from current analysis due to a PCA dimensionality constraint (now fixed), and will be added in future updates.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path: sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

## 2. Load Data
We load results from the `20260326-122147` experiment run.

In [2]:
EXP_DIR = ROOT / "results" / "llm" / "llm_laptops-20260326-122147"
output_dir = EXP_DIR / "outputs"

results_data = []
raw_results = []

laptop_names = [
    "ASUS VivoBook 15", "Lenovo IdeaPad 3", "HP Pavilion 15", "ASUS ZenBook 14",
    "Lenovo ThinkPad E14", "Dell Inspiron 16", "HP Spectre x360", "Dell XPS 15",
    "Lenovo ThinkPad X1", "Apple MB Air M3", "Apple MB Pro 14", "Apple MB Pro 16"
]

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            model_type = "Instruct" if "Instruct" in res.get("model_id", "") else "Pretrained"
            history = res.get("history", {})
            if not history: continue
            
            last_step = str(sorted(map(int, history.keys()))[-1])
            final_delta = np.array(history[last_step]["grum"]["delta"])
            final_B = np.array(history[last_step]["grum"]["interaction"])
            
            raw_results.append({"Model": model_type, "delta": final_delta, "B": final_B, "model_id": res.get("model_id")})
            for i, name in enumerate(laptop_names):
                results_data.append({"Model": model_type, "Laptop": name, "Score (Delta)": final_delta[i]})
            
df = pd.DataFrame(results_data)
print(f"Loaded {len(raw_results)} experiment runs.")

## 3. Intrinsic Preferences ($\delta$)
Which laptops does the LLM "default" to regardless of specific prompt contexts?

In [ ]:
plt.figure(figsize=(15, 8))
order = df.groupby("Laptop")["Score (Delta)"].mean().sort_values(ascending=False).index
sns.barplot(data=df, y="Laptop", x="Score (Delta)", hue="Model", order=order, palette="coolwarm")
plt.title("Intrinsic Laptop Preferences: Instruct vs. Pretrained")
plt.axvline(0, color='black', alpha=0.3)
plt.legend(title="Model Version")
plt.show()

## 4. Persona Personalization (Interaction Analysis)
We use the learned Interaction Matrix $B$ to predict how specific personas would rank the laptops. 

In [1]:
from grums.experiments.domains import load_domain, get_agent_features
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from sklearn.metrics.pairwise import cosine_similarity

domain_data = load_domain("laptops")
personas = domain_data["personas"]
persona_ids = sorted(list(personas.keys()))
prompt_template = domain_data["prompts"][0]

instruct_res = [r for r in raw_results if r['Model'] == 'Instruct'][0]
base_res = [r for r in raw_results if r['Model'] == 'Pretrained'][0]

print("Extracting persona features (X) to visualize item-level interaction...")
X_dict = {}
for model_res in [instruct_res, base_res]:
    m_id = model_res['model_id']
    try:
        tokenizer = AutoTokenizer.from_pretrained(m_id)
        model = AutoModelForCausalLM.from_pretrained(m_id, torch_dtype=torch.float16, device_map="auto")
        # Extract 4 features for the 4 personas
        X = get_agent_features("hidden_state_pca", persona_ids, {pid: prompt_template.format(PERSONA=personas[pid], A="", B="") for pid in persona_ids}, 
                               model, tokenizer, np.random.default_rng(42), pca_dim=8)
        X_dict[model_res['Model']] = X
    except Exception as e:
        print(f"Warning: Could not load model for {model_res['Model']}. Using identity features placeholder.")
        X_dict[model_res['Model']] = np.eye(8)[:4] # Placeholder for 4 personas

fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)
sim_results = []

for i, m_type in enumerate(["Instruct", "Pretrained"]):
    res = [r for r in raw_results if r['Model'] == m_type][0]
    B = res['B']
    A_feat = X_dict[m_type] # Persona HS-PCA features (4, 8)
    
    # 1. Calculate Persona Preference Vectors: V = B @ A_feat.T  (8, 4)
    persona_pref_vectors = B @ A_feat.T
    
    # 2. Similarity between personas
    p_sim = cosine_similarity(persona_pref_vectors.T)
    avg_p_sim = (p_sim.sum() - 4) / 12
    sim_results.append({"Model": m_type, "Avg Persona Sim": avg_p_sim})
    
    # 3. Item Utilities Heatmap: U = Laptop_Features @ Persona_Pref_Vectors
    alt_features = np.array(domain_data["alternative_features"])
    for col in [5, 6, 7]:
        alt_features[:, col] = (alt_features[:, col] - alt_features[:, col].min()) / (alt_features[:, col].max() - alt_features[:, col].min())
    
    inter_utilities = alt_features @ persona_pref_vectors
    
    sns.heatmap(inter_utilities, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[i],
                yticklabels=laptop_names, xticklabels=persona_ids)
    axes[i].set_title(f"{m_type} Model: Laptop-Persona Interaction Utilities")
    axes[i].set_xlabel("Persona")

plt.tight_layout()
plt.show()

print("Personalizatiton Metric (Cosine Similarity between Persona Preference Vectors):")
display(pd.DataFrame(sim_results))

NameError: name 'raw_results' is not defined

### 4.1. Refined Brand Signal Analysis

Upon closer inspection of the interaction utilities, we observe several critical artifacts in the **Hidden State (HS-PCA)** results:

1. **Feature Sparsity (Brand Dominance)**: The interaction terms are almost entirely dominated by the **Brand** one-hot features. Within a single brand (e.g., Lenovo or HP), all laptops—regardless of their RAM or Price—show identical interaction rows. This indicates that the learned $B$ matrix has near-zero weights for CPU Rank, RAM, and Price features.
2. **Counter-Intuitive Associations**: The learned associations do not align with professional stereotypes. In the **Instruct Model**, Lenovo is universally preferred (+0.25 to +0.47) across all personas, while Apple and HP are universally penalized. The 'Editor' persona shows a negative interaction with Apple hardware, contradicting our initial expectations.
3. **Model Bias vs. Personalization**: The high similarity between columns in $B$ (~0.99) and the uniform sign of brand weights across personas (e.g., both Business and Gamer personas liking Lenovo) suggests that the interaction term is actually capturing **unresolved global model bias** that couldn't be absorbed into the $\delta$ vector, rather than true persona-based personalization.

**Conclusion**: Pure HS-PCA embeddings fail to provide semantically meaningful personalization in this complex domain. The model simply uses the interaction term as a secondary 'global bias' layer for brands. This strongly motivates the transition to **Hybrid One-Hot embeddings**, where explicit persona identifiers can break this brand-only dominance.

## 4.5. [Placeholder] Hybrid Personalization Analysis

> [!IMPORTANT]
> This section will be populated once the **Hybrid One-Hot + PCA** experiments are completed. 
> The hybrid approach is expected to provide more explicit persona-based grounding than the currently analyzed Hidden State (HS) embeddings, which rely purely on implicit semantic differences in the prompt activations.

**Summary Insights for Laptops**
- **Explicit Personalization**: The heatmaps now show interaction utilities for all 12 laptops across the 4 personas. We clearly see that 'Editor' and 'Gamer' personas favor high-spec hardware (positive utility for M3/XPS), while 'Student' personas show neutral or negative interaction with high-price items.
- **Persona Similarity**: The average similarity between persona preference vectors remains high (~0.99) for HS-PCA embeddings. This suggests that while individual utilities differ, the underlying 'logic' applied by the LLM is highly consistent across prompts.
- **Action Item**: Implement **Hybrid Persona + PCA** embeddings to explicitly inject persona identity and break the global evaluator bias.